# GHIM Energy Module — Proposal Figures

Generate key figures for the proposal slide deck.
Saves PNGs to `../slides/figures/`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

FIGDIR = Path('../slides/figures')
FIGDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 200,
    'font.size': 11,
    'axes.titlesize': 13,
    'figure.facecolor': 'white',
})

In [8]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 4.5))

# Example 3-grade supply curve (like oil)
grades = [
    {"available": 20, "cost": 4.0},   # Grade 1: cheap
    {"available": 30, "cost": 7.0},   # Grade 2: mid
    {"available": 50, "cost": 12.0},  # Grade 3: expensive
]

# Piecewise-linear supply: production_at_price(p)
prices = np.linspace(0, 15, 500)
production = []
for p in prices:
    total = 0
    for i, g in enumerate(grades):
        if p >= g["cost"]:
            total += g["available"]
        else:
            prev_cost = grades[i-1]["cost"] if i > 0 else 0.0
            span = g["cost"] - prev_cost
            if span > 0 and p > prev_cost:
                total += g["available"] * (p - prev_cost) / span
            break
    production.append(total)

# Also draw step-function (naive) for comparison
prod_step = []
for p in prices:
    total = sum(g["available"] for g in grades if p >= g["cost"])
    prod_step.append(total)

ax.plot(production, prices, '-', lw=2.5, color='#2196F3', label='Piecewise-linear (GHIM)')
ax.plot(prod_step, prices, '--', lw=1.5, color='#999', label='Step function (naive)')

# Grade boundaries
cumul = 0
for g in grades:
    cumul += g["available"]
    ax.axhline(y=g["cost"], color='#ddd', lw=0.8, zorder=0)
    ax.plot(cumul, g["cost"], 'o', ms=7, color='#F44336', zorder=5)
    ax.annotate(f'Grade {grades.index(g)+1}\n{g["available"]} EJ (${g["cost"]}/GJ)',
                xy=(cumul, g["cost"]), xytext=(cumul+5, g["cost"]-2),
                fontsize=8, color='#666',
                arrowprops=dict(arrowstyle='->', color='#999', lw=0.8))

ax.set_xlabel('Production (EJ)', fontsize=12)
ax.set_ylabel('Price ($/GJ)', fontsize=12)
ax.set_title('Grade-Based Supply Curve: Piecewise-Linear Interpolation', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0, 110)
ax.set_ylim(0, 15)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig('../slides/figures/supply_curve.png', dpi=200, bbox_inches='tight')
plt.show()

/tmp/ipykernel_52371/2645984971.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1. Run the Model

In [2]:
from ghim.data.ssp import load_ssp_data
from ghim.solver.recursive import run_model
from ghim.output.reporting import results_to_dataframe

ssp_data = load_ssp_data('SSP2')
results = run_model(ssp_data, 'SSP2', policy=None, trade_enabled=True)
df = results_to_dataframe(results)
print(f'Results: {len(df)} rows, years {df.year.min()}-{df.year.max()}')

Results: 310 rows, years 2000-2150


## 2. Architecture Diagram

In [3]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis('off')

# Color palette
C_ECON = '#2196F3'    # blue
C_ENERGY = '#FF9800'  # orange
C_TRADE = '#4CAF50'   # green
C_POLICY = '#9C27B0'  # purple
C_FEED = '#F44336'    # red

box_kw = dict(boxstyle='round,pad=0.4', linewidth=1.5)

def draw_box(ax, x, y, text, color, w=2.0, h=0.7):
    rect = mpatches.FancyBboxPatch((x - w/2, y - h/2), w, h,
                                    boxstyle='round,pad=0.15',
                                    facecolor=color, edgecolor='#333',
                                    alpha=0.85, linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=10,
            fontweight='bold', color='white')

def draw_arrow(ax, x1, y1, x2, y2, color='#666', lw=1.5):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw))

# Economic core (top)
draw_box(ax, 2.5, 6.0, 'SSP Scenarios\n(Pop, GDP ref)', '#607D8B', w=2.2)
draw_box(ax, 5.0, 6.0, 'KLEM / DICE\nGDP = A·K\u1d45·L\u00b9\u207b\u1d45', C_ECON, w=2.2)

# Energy supply chain (middle)
draw_box(ax, 2.0, 4.0, 'Electricity\n(8 techs)', C_ENERGY)
draw_box(ax, 5.0, 4.0, 'Refining &\nHydrogen', C_ENERGY)
draw_box(ax, 8.0, 4.0, 'Final Demand\n(3 sectors)', C_ENERGY)

# Choice mechanisms (middle-bottom)
draw_box(ax, 2.0, 2.5, 'Logit +\nStock Turnover', '#795548', w=2.0, h=0.6)
draw_box(ax, 5.0, 2.5, 'Learning\nCurves', '#795548', w=2.0, h=0.6)

# Trade (bottom left)
draw_box(ax, 2.0, 1.0, 'Trade Clearing\n(coal, oil, gas)', C_TRADE, w=2.2)

# Policy (bottom right)
draw_box(ax, 8.0, 2.5, 'Policy\n(8 instruments)', C_POLICY, w=2.0, h=0.6)

# Feedback
draw_box(ax, 8.0, 1.0, 'Energy Cost\nFeedback', C_FEED, w=2.0, h=0.6)

# Arrows: economic -> energy
draw_arrow(ax, 3.6, 6.0, 4.0, 6.0, C_ECON)
draw_arrow(ax, 5.0, 5.6, 5.0, 4.4, C_ECON)
draw_arrow(ax, 5.0, 5.6, 2.0, 4.4, C_ECON)
draw_arrow(ax, 5.0, 5.6, 8.0, 4.4, C_ECON)

# Arrows: supply chain flow
draw_arrow(ax, 3.0, 4.0, 4.0, 4.0, C_ENERGY)
draw_arrow(ax, 6.0, 4.0, 7.0, 4.0, C_ENERGY)

# Arrows: choice & learning
draw_arrow(ax, 2.0, 2.8, 2.0, 3.6, '#795548')
draw_arrow(ax, 5.0, 2.8, 5.0, 3.6, '#795548')

# Arrows: trade
draw_arrow(ax, 2.0, 1.4, 2.0, 2.2, C_TRADE)

# Arrows: policy
draw_arrow(ax, 8.0, 2.8, 8.0, 3.6, C_POLICY)

# Feedback loop
draw_arrow(ax, 8.0, 3.6, 8.0, 1.4, C_FEED)
draw_arrow(ax, 9.0, 1.0, 9.5, 1.0, C_FEED)
ax.annotate('', xy=(9.5, 6.0), xytext=(9.5, 1.0),
            arrowprops=dict(arrowstyle='->', color=C_FEED, lw=1.5))
ax.annotate('', xy=(6.2, 6.0), xytext=(9.5, 6.0),
            arrowprops=dict(arrowstyle='->', color=C_FEED, lw=1.5))

# Legend
ax.text(5.0, 0.2, '10 AR6 R10 Regions  |  2000\u20132150  |  5-year timesteps  |  Recursive-dynamic',
        ha='center', va='center', fontsize=9, color='#666', style='italic')

ax.set_title('GHIM Energy Module — System Architecture', fontsize=14, fontweight='bold', pad=10)
fig.tight_layout()
fig.savefig(FIGDIR / 'architecture_diagram.png', bbox_inches='tight')
plt.close(fig)
print('Saved architecture_diagram.png')

Saved architecture_diagram.png


## 3. GDP Trajectory (Model vs SSP Reference)

In [4]:
global_gdp = df.groupby('year').agg(
    model_gdp=('gdp_billion_usd', 'sum'),
    ssp_gdp=('ssp_reference_gdp_billion_usd', 'sum'),
).reset_index()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(global_gdp['year'], global_gdp['ssp_gdp'] / 1000, 'k--', lw=2, label='SSP2 Reference')
ax.plot(global_gdp['year'], global_gdp['model_gdp'] / 1000, '-', lw=2, color='#2196F3', label='GHIM Model')
ax.set_xlabel('Year')
ax.set_ylabel('Global GDP (Trillion 2020$ PPP)')
ax.set_title('Global GDP: GHIM Endogenous vs SSP2 Reference')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(2000, 2150)
fig.tight_layout()
fig.savefig(FIGDIR / 'gdp_trajectory.png', bbox_inches='tight')
plt.close(fig)
print('Saved gdp_trajectory.png')

Saved gdp_trajectory.png


## 4. Electricity Generation Mix (Stacked Area)

In [5]:
elec_cols = [c for c in df.columns if c.startswith('elec_') and c.endswith('_ej')]
elec_global = df.groupby('year')[elec_cols].sum()

# Rename columns for display
rename = {c: c.replace('elec_', '').replace('_ej', '').replace('_', ' ').title() for c in elec_cols}
elec_display = elec_global.rename(columns=rename)

# Order by total generation (largest at bottom)
order = elec_display.sum().sort_values(ascending=False).index.tolist()
elec_display = elec_display[order]

# Color map
tech_colors = {
    'Coal': '#555555', 'Gas Cc': '#FF9800', 'Oil': '#795548',
    'Nuclear': '#9C27B0', 'Hydro': '#03A9F4',
    'Wind': '#4CAF50', 'Solar': '#FFC107', 'Biomass': '#8BC34A',
}
colors = [tech_colors.get(t, '#999') for t in order]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.stackplot(elec_display.index, *[elec_display[col] for col in order],
             labels=order, colors=colors, alpha=0.85)
ax.set_xlabel('Year')
ax.set_ylabel('Electricity Generation (EJ)')
ax.set_title('Global Electricity Generation Mix (SSP2)')
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.set_xlim(2000, 2150)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGDIR / 'electricity_mix.png', bbox_inches='tight')
plt.close(fig)
print('Saved electricity_mix.png')

Saved electricity_mix.png


## 5. Global CO2 Emissions Trajectory

In [6]:
global_em = df.groupby('year')['emissions_mtco2'].sum().reset_index()
global_em['gt_co2'] = global_em['emissions_mtco2'] / 1000

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(global_em['year'], global_em['gt_co2'], '-', lw=2.5, color='#F44336')
ax.fill_between(global_em['year'], 0, global_em['gt_co2'], alpha=0.15, color='#F44336')
ax.axhline(y=0, color='#333', lw=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Global CO\u2082 Emissions (GtCO\u2082/yr)')
ax.set_title('Global CO\u2082 Emissions Trajectory (SSP2, No Policy)')
ax.grid(alpha=0.3)
ax.set_xlim(2000, 2150)
fig.tight_layout()
fig.savefig(FIGDIR / 'emissions_trajectory.png', bbox_inches='tight')
plt.close(fig)
print('Saved emissions_trajectory.png')

Saved emissions_trajectory.png


## 6. World Fuel Prices (Trade)

In [7]:
# World prices are the same across regions in each year (global clearing)
# Take first region's prices
first_region = df['region'].iloc[0]
prices_df = df[df['region'] == first_region][['year',
    'world_price_coal_usd_gj', 'world_price_oil_usd_gj', 'world_price_gas_usd_gj'
]].copy()
prices_df = prices_df[prices_df['year'] >= 2020]  # trade only from 2025

fuel_colors = {'coal': '#555', 'oil': '#795548', 'gas': '#FF9800'}

fig, ax = plt.subplots(figsize=(8, 4.5))
for fuel, color in fuel_colors.items():
    col = f'world_price_{fuel}_usd_gj'
    ax.plot(prices_df['year'], prices_df[col], '-o', ms=3, lw=2,
            color=color, label=fuel.capitalize())

ax.set_xlabel('Year')
ax.set_ylabel('World Price (2020$/GJ)')
ax.set_title('World Fossil Fuel Prices (SSP2, Trade-Cleared)')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(2020, 2150)
fig.tight_layout()
fig.savefig(FIGDIR / 'trade_world_prices.png', bbox_inches='tight')
plt.close(fig)
print('Saved trade_world_prices.png')

Saved trade_world_prices.png


## 7. Net Exports by Region (Oil)

In [8]:
# Focus on oil net exports for a few key regions
key_regions = ['Middle East', 'North America', 'Europe', 'Eastern Asia', 'Southern Asia', 'Africa']
oil_exports = df[df['region'].isin(key_regions)].pivot_table(
    values='net_exports_oil_ej', index='year', columns='region'
)
oil_exports = oil_exports[oil_exports.index >= 2025]

region_colors = {
    'Middle East': '#F44336', 'North America': '#2196F3', 'Europe': '#4CAF50',
    'Eastern Asia': '#FF9800', 'Southern Asia': '#9C27B0', 'Africa': '#795548',
}

fig, ax = plt.subplots(figsize=(8, 4.5))
for region in key_regions:
    if region in oil_exports.columns:
        ax.plot(oil_exports.index, oil_exports[region], '-o', ms=3, lw=1.8,
                color=region_colors.get(region, '#999'), label=region)

ax.axhline(y=0, color='#333', lw=0.8, ls='--')
ax.set_xlabel('Year')
ax.set_ylabel('Net Oil Exports (EJ)')
ax.set_title('Oil Net Exports by Region (SSP2)')
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
ax.set_xlim(2025, 2150)
fig.tight_layout()
fig.savefig(FIGDIR / 'trade_net_exports.png', bbox_inches='tight')
plt.close(fig)
print('Saved trade_net_exports.png')
print('\nAll figures saved to slides/figures/')

Saved trade_net_exports.png

All figures saved to slides/figures/
